# Trabalho Prático: Aprendizado de Máquina (Supervisionado e Não Supervisionado)
**Dataset:** QSAR Biodegradation (1054 amostras, 41 features moleculares)
**Objetivo:** Classificação binária de substâncias químicas (Prontamente Biodegradável - RB vs Não Prontamente Biodegradável - NRB).

## Estrutura do Notebook:
1. Configuração e Importação de Bibliotecas
2. Leitura, Limpeza e Normalização Obrigatória dos Dados
3. Divisão dos Dados (70% Treinamento / 15% Validação / 15% Teste)
4. Modelagem Supervisionada (Decision Tree, KNN, Artificial Neural Network) com otimização
5. Modelagem Não Supervisionada (K-Means) com Método do Cotovelo e Silhouette Score
6. Análise de Componentes Principais (PCA) para visualização dos Clusters

In [ ]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento e Divisão
from sklearn.model_selection import train_test_split, PredefinedSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Modelos Supervisionados
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

# Clusterização
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Métricas de Avaliação
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score, silhouette_score)

from imblearn.over_sampling import SMOTE

# Configurações visuais
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
import warnings
warnings.filterwarnings('ignore') # Ignorar avisos de depreciação menores

## 1. Carregamento, Limpeza e Normalização dos Dados
Mesmo que o arquivo original já contenha a palavra "normalizado", passaremos os dados por um pipeline rigoroso de verificação de nulos, separação de variáveis e padronização estatística (z-score). O resultado será exportado para um novo arquivo CSV limpo, cumprindo o requisito de ter um script de pré-processamento.

In [ ]:
# 1.1 Carregamento dos dados
# Lendo o arquivo original (certifique-se de que o arquivo está no mesmo diretório do notebook)
# Assumindo separador por vírgula com base nos metadados da amostra
df_raw = pd.read_csv('biodeg_normalizado.csv')

print(f"Dimensões originais do dataset: {df_raw.shape}")

# 1.2 Limpeza de Dados (Tratamento de Nulos/Duplicatas)
# Remoção de duplicatas se existirem
df_raw = df_raw.drop_duplicates()
# Remoção de valores nulos (se existirem)
df_raw = df_raw.dropna()
print(f"Dimensões após limpeza básica: {df_raw.shape}")

# 1.3 Separação de Features (X) e Target (y)
target_col = 'experimental class' # Conforme inspecionado no cabeçalho dos dados
X_raw = df_raw.drop(columns=[target_col])
y_raw = df_raw[target_col]

# 1.4 Codificação da Variável Alvo
# RB (Ready Biodegradable) e NRB (Not Ready Biodegradable)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)
# Mapeamento para conferência: classes originais para valores numéricos
le_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print(f"Mapeamento do Target: {le_mapping}")

# 1.5 Normalização Estatística (Z-Score)
# É crucial para o KNN, K-Means e Redes Neurais que as features estejam na mesma escala
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), columns=X_raw.columns)

# 1.6 Salvando o Dataset Final Processado
df_clean = X_scaled.copy()
df_clean['target_encoded'] = y_encoded
df_clean.to_csv('biodeg_limpo_e_processado.csv', index=False)
print("\nDataset limpo e normalizado salvo com sucesso como 'biodeg_limpo_e_processado.csv'.")

## 2. Divisão dos Dados (Holdout Estratificado: 70/15/15)
O dataset é desbalanceado (~66% NRB vs ~34% RB). Portanto, a estratificação é vital para garantir que a mesma proporção de classes exista no treino, validação e teste.

In [ ]:
# Para obter 70% Treino, 15% Validação e 15% Teste:
# Primeiro passo: Extraímos 30% do total para um conjunto temporário (que será dividido ao meio)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y_encoded,
    test_size=0.30,
    stratify=y_encoded,
    random_state=42
)

# Segundo passo: Dividimos os 30% temporários em 50/50, resultando em 15% de validação e 15% de teste reais
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print(f"Tamanho do Conjunto de Treino:    {X_train.shape[0]} amostras ({X_train.shape[0]/df_clean.shape[0]*100:.1f}%)")
print(f"Tamanho do Conjunto de Validação: {X_val.shape[0]} amostras ({X_val.shape[0]/df_clean.shape[0]*100:.1f}%)")
print(f"Tamanho do Conjunto de Teste:     {X_test.shape[0]} amostras ({X_test.shape[0]/df_clean.shape[0]*100:.1f}%)")

# =====================================================================
# APLICAÇÃO DO SMOTE (Apenas no conjunto de TREINO)
# =====================================================================
print("\n--- Aplicando SMOTE no conjunto de Treino ---")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Verificação do novo balanceamento
unique_smote, counts_smote = np.unique(y_train_smote, return_counts=True)
print(f"Novo tamanho do Treino (após SMOTE): {X_train_smote.shape[0]} amostras")
print(f"Nova proporção das classes no Treino: {dict(zip(unique_smote, counts_smote/len(y_train_smote)))}")
# =====================================================================

# Verificação do balanceamento no teste
unique, counts = np.unique(y_test, return_counts=True)
print(f"\nProporção das classes no Teste: {dict(zip(unique, counts/len(y_test)))}")

## 3. Modelagem Supervisionada (Classificação)
Testaremos 3 algoritmos:
1. **Decision Tree (Árvore de Decisão)**
2. **KNN (K-Nearest Neighbors)**
3. **Artificial Neural Network (Multi-Layer Perceptron)**

Para garantir o uso *exato* do conjunto de Validação para a escolha dos melhores hiperparâmetros (e não usar cross-validation interna no treino), utilizaremos o `PredefinedSplit`.

In [ ]:
# Construindo o PredefinedSplit para forçar o GridSearchCV a usar nosso X_val específico
# Concatenamos Treino e Validação
X_train_val = pd.concat([X_train_smote, X_val], axis=0)
y_train_val = np.concatenate([y_train_smote, y_val])

# -1 indica TREINO (agora com amostras sintéticas), 0 indica VALIDAÇÃO (intocada)
test_fold = np.zeros(X_train_val.shape[0])
test_fold[:X_train_smote.shape[0]] = -1
ps = PredefinedSplit(test_fold)

# Dicionário para armazenar a performance dos modelos no conjunto de teste final
resultados_modelos = {}

def avaliar_modelo(nome, modelo, params):
    """
    Função utilitária para buscar melhores parâmetros com o conjunto de Validação,
    treinar o modelo final e avaliá-lo no conjunto de Teste.
    """
    print(f"--- Treinando e Otimizando: {nome} ---")
    grid = GridSearchCV(modelo, params, cv=ps, scoring='f1_macro', n_jobs=-1)
    grid.fit(X_train_val, y_train_val)

    best_model = grid.best_estimator_
    print(f"Melhores Hiperparâmetros encontrados: {grid.best_params_}")

    # Previsão no conjunto de TESTE (nunca visto até agora)
    y_pred = best_model.predict(X_test)

    # Métricas
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    resultados_modelos[nome] = {'Acurácia': acc, 'F1-Score (Macro)': f1}

    print("\nRelatório de Classificação no Conjunto de TESTE:")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

    # Plotagem da Matriz de Confusão
    plt.figure(figsize=(5,4))
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
    plt.title(f'Matriz de Confusão - {nome}')
    plt.ylabel('Classe Real')
    plt.xlabel('Classe Predita')
    plt.show()
    print("="*60 + "\n")
    return best_model